# Week 7: Beat the 39 — Price prediction (IbrahimSheriff2)

**Goal:** Get average absolute error **below 39.85** (ideally into the lower 30s).  
**Metric:** Same as instructor — average $ error on 250 test samples.  
**HF user:** `sheriff`

In [ ]:
%pip install -q pandas scikit-learn datasets transformers torch peft bitsandbytes trl accelerate matplotlib python-dotenv

In [ ]:
import os
import re
import math
from pathlib import Path
from datetime import datetime
import torch
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
import matplotlib.pyplot as plt

load_dotenv(override=True)
set_seed(42)
%matplotlib inline

## Config (hyperparameters to tune)

**Instructor baseline:** 39.85. Try varying these to get into the lower 30s.  
**Tip:** Data manipulation (filtering, balancing, prompt format) often gives the biggest gain.

In [ ]:
HF_USER = "sheriff"
DATASET_NAME = "ed-donner/pricer-data"
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "pricer"

# --- Hyperparameters (tune these to beat 39.85) ---
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 256
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

# Optional: subsample for quick experiments (set to None to use full train)
TRAIN_SUBSAMPLE = None  # e.g. 20000

EVAL_SIZE = 250  # same as instructor Tester
hf_token = os.environ.get("HF_TOKEN")

## Load data

In [ ]:
dataset = load_dataset(DATASET_NAME, token=hf_token)
train_raw = dataset["train"]
test_raw = dataset["test"]
print(f"Train: {len(train_raw)}, Test: {len(test_raw)}")
print("Columns:", train_raw.column_names)
if len(train_raw) > 0:
    ex = train_raw[0]
    print("Sample keys:", list(ex.keys()))
    if "text" in ex:
        print("Sample text (first 300 chars):", (ex["text"] or "")[:300])

## Prepare training data (prompt + completion)

**Data manipulation idea:** You can filter by price range, oversample rare buckets, or clean `text` here to teach the model better.

In [ ]:
def build_train_text(example):
    # pricer-data: "text" is the prompt; model should complete with "Price is $X"
    prompt = (example.get("text") or "").strip()
    price = example.get("price")
    if price is None:
        return None
    try:
        p = float(price)
    except (TypeError, ValueError):
        return None
    # Completion format expected at eval (extract_price looks for "Price is $")
    completion = f"Price is ${p:.2f}"
    return prompt + completion

train_list = []
for i in range(len(train_raw)):
    row = train_raw[i]
    text = build_train_text(row)
    if text:
        train_list.append({"text": text})

if TRAIN_SUBSAMPLE:
    np.random.seed(42)
    idx = np.random.choice(len(train_list), min(TRAIN_SUBSAMPLE, len(train_list)), replace=False)
    train_list = [train_list[i] for i in idx]

train_ds = Dataset.from_list(train_list)
print(f"Training samples: {len(train_ds)}")

In [ ]:
# Validation: small subset from train for eval_strategy
val_size = min(500, len(train_ds) // 10)
val_ds = train_ds.select(range(val_size))
train_ds = train_ds.select(range(val_size, len(train_ds)))
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## Model & QLoRA

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.generation_config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Completion-only loss: only tokens after "Price is $" are trained
RESPONSE_TEMPLATE = "Price is $"
collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)

In [ ]:
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
OUTPUT_DIR = f"{PROJECT_NAME}-{RUN_NAME}"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="steps",
    eval_steps=200,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_32bit",
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
    data_collator=collator,
)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

## Evaluation — beat 39.85

Load the saved adapter (or set `ADAPTER_PATH` to a previous run) and run the same metric as the instructor: **average absolute error** on 250 test samples.

In [ ]:
ADAPTER_PATH = OUTPUT_DIR  # or e.g. "pricer-2025-03-09_12.00.00"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
def extract_price(s):
    if "Price is $" in s:
        contents = s.split("Price is $")[1].replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", contents)
        return float(m.group()) if m else 0.0
    return 0.0

@torch.no_grad()
def predict(prompt, max_new_tokens=15):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return extract_price(full)

In [ ]:
GREEN, YELLOW, RED, RESET = "\033[92m", "\033[93m", "\033[91m", "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

class Tester:
    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.title = title or "Model"
        self.size = min(size, len(data))
        self.guesses, self.truths, self.errors, self.sles, self.colors = [], [], [], [], []

    def color_for(self, error, truth):
        if error < 40 or (truth and error / truth < 0.2):
            return "green"
        if error < 80 or (truth and error / truth < 0.4):
            return "orange"
        return "red"

    def run_datapoint(self, i):
        row = self.data[i]
        prompt = (row.get("text") or "").strip()
        truth = float(row.get("price", 0))
        guess = self.predictor(prompt)
        error = abs(guess - truth)
        log_err = math.log(truth + 1) - math.log(guess + 1)
        sle = log_err ** 2
        color = self.color_for(error, truth)
        title = (prompt[:50] + "...") if len(prompt) > 50 else prompt
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:.2f} {title}{RESET}")

    def report(self):
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for c in self.colors if c == "green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:.2f} Hits={hits/self.size*100:.1f}%"
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color="deepskyblue", lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel("Ground Truth")
        plt.ylabel("Model Estimate")
        plt.title(title)
        plt.show()
        return average_error

    def run(self):
        for i in range(self.size):
            self.run_datapoint(i)
        return self.report()

    @classmethod
    def test(cls, predictor, data, title=None, size=250):
        t = cls(predictor, data, title=title, size=size)
        return t.run()

In [ ]:
test_data = test_raw.select(range(EVAL_SIZE))
avg_error = Tester.test(predict, test_data)
print(f"\n>>> Average error: ${avg_error:,.2f}  (instructor baseline: 39.85; goal: < 39, lower 30s)")